In [1]:
import numpy as np
import pandas as pd
import csv
import time
import chardet

In [2]:


sales_name = "서울시_행정동_매출_월단위_202406_202512.csv"
spatiotemporal_name = "서울시_행정동_시공간_월단위_202408_202603.csv"
global_name = "서울은행_글로발변수_월단위_202406_202602.csv"
apt_name = "서울시_행정동_아파트가격_월단위_202406_202512.csv"


people_name = "서울시_행정동_인구_월단위_202406_202512.csv"

data_name = "서울시_행정동_생활비용_월단위_202408_202512.csv"


In [30]:
# 2개식 읽고 파일 합치기

sales_df = pd.read_csv(sales_name, encoding="utf-8-sig")

spatiotemporal_df = pd.read_csv(spatiotemporal_name, encoding="utf-8-sig")


sales_df = sales_df[(sales_df["YYYYMM"] >= 202408) & (sales_df["YYYYMM"] <= 202512)]

spatiotemporal_df = spatiotemporal_df[(spatiotemporal_df["YYYYMM"] >= 202408) & (spatiotemporal_df["YYYYMM"] <= 202512)]

merge_df = pd.merge(sales_df, spatiotemporal_df, on=["YYYYMM","행정동코드"], how="outer")

print(len(merge_df))
merge_df.head(2)



7276


,기준_년분기_코드,YYYYMM,행정동코드,당월_매출_금액,당월_매출_건수,주중_매출_금액,주말_매출_금액,남성_매출_금액,여성_매출_금액,행정동이름,AREA_M2,LAT,LON,전체승객수,지하철승객수,버스승객수
0,20242.0,202408,1111051500,2.582955e+10,1218354.0,1.931113e+10,6.518425e+09,8.443738e+09,1.532099e+10,청운효자동,2438307.0,126.97042,37.58466,252717.0,0.0,252717.0
1,20242.0,202408,1111053000,1.028522e+11,4456047.0,8.339905e+10,1.945317e+10,4.075866e+10,3.963525e+10,사직동,1165780.0,126.97014,37.57411,2375818.0,1557165.0,818653.0


In [31]:


merge_df = merge_df.drop(columns=["기준_년분기_코드"])


apt_df = pd.read_csv(apt_name, encoding="utf-8-sig")

apt_df = apt_df[(apt_df["YYYYMM"] >= 202408) & (apt_df["YYYYMM"] <= 202512)]

merge_df = pd.merge(merge_df, apt_df, on=["YYYYMM","행정동코드"], how="outer")

global_df = pd.read_csv(global_name, encoding="utf-8-sig")

global_df = global_df[(global_df["YYYYMM"] >= 202408) & (global_df["YYYYMM"] <= 202512)]

merge_df = pd.merge(merge_df, global_df, on=["YYYYMM"], how="outer")


In [32]:


merge_df.to_csv(data_name, encoding="utf-8-sig", index=False)



In [7]:
merge_df = pd.read_csv("서울시_행정동_생활비용_임시_202408_202512.csv", encoding="utf-8-sig")

people_df = pd.read_csv(people_name, encoding="utf-8-sig")

people_df = people_df[(people_df["YYYYMM"] >= 202408) & (people_df["YYYYMM"] <= 202512)]

people_df["행정동코드"] = people_df["행정동코드"] * 100

merge_df = pd.merge(merge_df, people_df, on=["YYYYMM","행정동코드"], how="outer")

merge_df.to_csv(data_name, encoding="utf-8-sig", index=False)

In [34]:
# 인구 추가

all_df = pd.read_csv("생활비용_모델용_월단위_202408_202512.csv",
                     #encoding="cp949")
                     encoding="utf-8-sig")


#all_df = all_df.dropna(how="all")

#all_df.drop(["기준_년분기_코드","행정동_코드_명"],axis=1,inplace=True)

all_df.to_csv(data_name, encoding="utf-8-sig", index=False)

all_df.isnull().sum()



YYYYMM                    0
행정동코드                     0
당월_매출_금액                  0
당월_매출_건수                  0
주중_매출_금액                  0
주말_매출_금액                  0
남성_매출_금액                  0
여성_매출_금액                  0
행정동이름                     0
AREA_M2                   0
LAT                       0
LON                       0
전체승객수                     0
지하철승객수                    0
버스승객수                     0
아파트_단지_수                  0
아파트_면적_66_제곱미터_미만_세대_수    0
아파트_면적_66_제곱미터_세대_수       0
아파트_면적_99_제곱미터_세대_수       0
아파트_면적_132_제곱미터_세대_수      0
아파트_면적_165_제곱미터_세대_수      0
아파트_평균_면적                 0
아파트_평균_시가                 0
M2                        0
KOSPI                     0
HOUSE_SALE                0
RENT_SALE                 0
FX                        0
생활인구합계                    0
dtype: int64

In [33]:
# 상일동 아파트 정보 읽어서 상일1동, 상일2동에 반반시 분할해서 넣기

sangil = 1174052000
sangil_one = 1174052500
sangil_two = 1174052600

all_df = pd.read_csv("서울시_행정동_생활비용_월단위_202408_202512.csv",
                     #encoding="cp949")
                     encoding="utf-8-sig")

apt_df = pd.read_csv(apt_name, encoding="utf-8-sig")

apt_df = apt_df[(apt_df["YYYYMM"] >= 202408) & (apt_df["YYYYMM"] <= 202512)]


value_cols = [
    "아파트_단지_수",
    "아파트_면적_66_제곱미터_미만_세대_수",
    "아파트_면적_66_제곱미터_세대_수",
    "아파트_면적_99_제곱미터_세대_수",
    "아파트_면적_132_제곱미터_세대_수",
    "아파트_면적_165_제곱미터_세대_수",
    "아파트_평균_면적",
    "아파트_평균_시가"
]

# 숫자형 변환
apt_df[value_cols] = (
    apt_df[value_cols]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
)

target_codes = [
    sangil_one,
    sangil_two
]

# source 데이터 추출
source_row = apt_df[apt_df["행정동코드"] == sangil]

# source 없으면 종료
if not source_row.empty:

    # 첫 row 기준
    source_values = source_row.iloc[0]

    # 절반 계산
    half_values = {
        col: source_values[col] / 2
        for col in value_cols
    }

    # target 행에 값 추가
    for target_code in target_codes:

        mask = all_df["행정동코드"] == target_code

        for col in value_cols:
            all_df.loc[mask, col] = (
                pd.to_numeric(all_df.loc[mask, col], errors="coerce")
                .fillna(0)
                + half_values[col]
            )

# 필요하면 source 제거
# df = df[df["행정동코드"] != source_code]

# 저장
all_df.to_csv("result.csv", index=False, encoding="utf-8-sig")



In [41]:
# 모두 숫자로 변환

cost_name = "생활비용_모델용_월단위_202408_202512.csv"

cost_df = pd.read_csv(cost_name, encoding="cp949")

key_cols = [
    "YYYYMM","행정동코드"
]

value_cols = [
    "당월_매출_금액",
    "당월_매출_건수",
    "주중_매출_금액",
    "주말_매출_금액",
    "남성_매출_금액",
    "여성_매출_금액",
    "AREA_M2",
    "LAT",
    "LON",
    "전체승객수",
    "지하철승객수",
    "버스승객수",
    "아파트_단지_수",
    "아파트_면적_66_제곱미터_미만_세대_수",
    "아파트_면적_66_제곱미터_세대_수",
    "아파트_면적_99_제곱미터_세대_수",
    "아파트_면적_132_제곱미터_세대_수",
    "아파트_면적_165_제곱미터_세대_수",
    "아파트_평균_면적",
    "아파트_평균_시가",
    "M2",
    "KOSPI",
    "HOUSE_SALE",
    "RENT_SALE",
    "FX",
    "생활인구합계"
]


for col in value_cols:

    # 문자열이면 쉼표 제거
    if cost_df[col].dtype == "object":
        cost_df[col] = (
            cost_df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.strip()
        )

    # 숫자 변환
    cost_df[col] = pd.to_numeric(
        cost_df[col],
        errors="coerce"
    )

print(cost_df[value_cols].dtypes)


cost_df.to_csv(cost_name, index=False, encoding="utf-8-sig")



당월_매출_금액                  float64
당월_매출_건수                  float64
주중_매출_금액                  float64
주말_매출_금액                  float64
남성_매출_금액                  float64
여성_매출_금액                  float64
AREA_M2                     int64
LAT                       float64
LON                       float64
전체승객수                       int64
지하철승객수                      int64
버스승객수                       int64
아파트_단지_수                    int64
아파트_면적_66_제곱미터_미만_세대_수      int64
아파트_면적_66_제곱미터_세대_수       float64
아파트_면적_99_제곱미터_세대_수         int64
아파트_면적_132_제곱미터_세대_수      float64
아파트_면적_165_제곱미터_세대_수      float64
아파트_평균_면적                 float64
아파트_평균_시가                   int64
M2                        float64
KOSPI                     float64
HOUSE_SALE                float64
RENT_SALE                 float64
FX                        float64
생활인구합계                      int64
dtype: object


In [4]:
# 테스트용 데이터 만들기

test_df = pd.read_csv("생활비용_모델용_테스트용_202501_202605_cli.csv", encoding="utf-8-sig")


test_df["YYYYMM"] = test_df["YYYYMM"].replace(202408, 202601)
test_df["YYYYMM"] = test_df["YYYYMM"].replace(202409, 202602)
test_df["YYYYMM"] = test_df["YYYYMM"].replace(202410, 202603)
test_df["YYYYMM"] = test_df["YYYYMM"].replace(202411, 202604)
test_df["YYYYMM"] = test_df["YYYYMM"].replace(202412, 202605)

test_df = test_df.sort_values(
    ["YYYYMM","행정동코드"]
)

test_df.to_csv("생활비용_모델용_테스트용_202501_202605_cli.csv", index=False, encoding="utf-8-sig")



In [4]:

base_df = pd.read_csv("생활비용_모델용_월단위_202408_202512_base.csv", encoding="utf-8-sig")

space_df = base_df.groupby("행정동코드")[["행정동코드","행정동이름","AREA_M2", "LAT", "LON"]].first()

space_df.to_csv("서울시_행정동_공간_base.csv", index=False, encoding="utf-8-sig")





In [8]:
# 서버용 csv용 신규 코드 매핑

edm_mapping_df = pd.read_csv("서울시_행정동ID_행정동코드_맵핑.csv", encoding="utf-8-sig")
master_df = pd.read_csv("서울시 행정동 마스터 정보.csv", encoding="cp949")

merge_df = pd.merge(
    edm_mapping_df,
    master_df,
    on="행정동_ID",
    how="left"
)

print(f"emd_mapping={len(edm_mapping_df)}, merge_df={len(merge_df)}")
merge_df.head(2)

new_df = merge_df[["행정동_ID","행정동코드","행정동_명칭","자치구_명칭"]]

new_df.to_csv("서울시_행정동ID_행정동코드_맵핑_base.csv", index=False, encoding="utf-8-sig")




emd_mapping=426, merge_df=426


In [15]:
# 서버용 csv 생성
edm_mapping_df = pd.read_csv("서울시_행정동ID_행정동코드_맵핑_base.csv", encoding="cp949")

test_df = pd.read_csv("생활비용_모델용_테스트용_202501_202605_cli.csv", encoding="utf-8-sig")

test_df["생활비용지수_등급"] = pd.cut(
    test_df["생활비용지수"],
    bins=5,              # min~max 자동 5등분
    labels=[5,4,3,2,1],
    include_lowest=True
)

test_df['년도']     = test_df['YYYYMM'] // 100
test_df['월']    = test_df['YYYYMM'] %  100

merge_df = pd.merge(
    test_df,
    edm_mapping_df,
    on="행정동코드",
    how="left"
)



new_df = merge_df[["YYYYMM","행정동코드","년도","월","행정동_명칭","자치구_명칭","생활비용지수_등급","생활비용지수"]]

new_df.to_csv("생활비용지수_테스트용_202501_202605.csv", index=False, encoding="utf-8-sig")







In [25]:
# 서버 테스트
def get_heatmap(year: int, month: int) -> pd.DataFrame:
    """
    추후 모델 또는 CSV가 들어오면 이 함수만 교체하면 됨.
    """

    try:
        df = pd.read_csv("생활비용지수.csv", encoding="utf-8-sig")

        df = df[(df["년도"] == year) & (df["월"] == month)]

        if df.empty:
            print(f"[expense] 데이터 없음: year={year}, month={month}")
            return pd.DataFrame(columns=["code", "dong", "gu", "grade", "score"])

        df = df.rename(columns={
            "행정동코드":"code",
            "행정동_명칭":"dong",
            "자치구_명칭":"gu",
            "생활비용지수_등급":"grade",
            "생활비용지수":"score"
            })

        return df[["code", "dong", "gu", "grade", "score"]].copy()

    except FileNotFoundError:
        print("[expense] 파일을 찾을 수 없습니다.")
        return pd.DataFrame(columns=["code", "dong", "gu", "grade", "score"])

    except UnicodeDecodeError:
        print("[expense] 인코딩 오류 발생")
        return pd.DataFrame(columns=["code", "dong", "gu", "grade", "score"])

    except pd.errors.EmptyDataError:
        print("[expense] 파일이 비어 있습니다.")
        return pd.DataFrame(columns=["code", "dong", "gu", "grade", "score"])

    except pd.errors.ParserError as e:
        print("[expense] CSV 파싱 오류:", e)
        return pd.DataFrame(columns=["code", "dong", "gu", "grade", "score"])

    except Exception as e:
        print("[expense] 알 수 없는 오류:", e)
        return pd.DataFrame(columns=["code", "dong", "gu", "grade", "score"])



df = get_heatmap(2026,5)

print(f"월데이터 크기 : {len(df)}")
df.head(2)


월데이터 크기 : 426


,code,dong,gu,grade,score
6816,1111051500,청운효자동,종로구,3,9.064569
6817,1111053000,사직동,종로구,2,9.597439
